# &#x1F913; CayleyPy Megaminx Solve Optimally: First steps 

🎉 Hello and welcome to the "CayleyPy Megaminx Solve Optimally" competition! &#x1F389;

This is a community competition, which is going to help to advance pathfinding methods on the puzzles like megaminx, as well as in other gropups. This notebook will show you how to load and interpret the data, how to use CayleyPy package to improve some of the paths and how make a valid submission file.

## &#x1F4E4; Loading and interpreting the data 
First, make sure that the competition's files are loaded as inputs. Once everything is at the right place, let's load the contents of **puzzle_info.json**

In [1]:
### Some useful imports ###

import json
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
INPUTS_DIR = Path("/kaggle/input/cayley-py-megaminx/")
with open(INPUTS_DIR/"puzzle_info.json", "r") as file:
    puzzle_info = json.load(file)

**puzzle_info.json** contains the central_state of the puzzle and the generators (aka "moves"). 



In [3]:
central_state = np.array(puzzle_info["central_state"])
generators = {k: np.array(v) for k, v in puzzle_info["generators"].items()}
print(f"central_state: {central_state}", end="\n\n")
print(f"Generator_names: {list(generators.keys())}")

central_state: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119]

Generator_names: ['U', '-U', 'D', '-D', 'F', '-F', 'B', '-B', 'L', '-L', 'DR', '-DR', 'BL', '-BL', 'FR', '-FR', 'BR', '-BR', 'FL', '-FL', 'R', '-R', 'DL', '-DL']


The central state is the state in which the puzzle is considered to be solved. All the initial states in evaluation can be obtained with some permutation of the central state.

Generators are the permutations which can be applied to a state of the puzzle. Each generator in **puzzle_info["generators"]** has an inverse one, so their combination doesn't affect the state. The inverse generators have a minus sign ("-") as the first symbol of their name.

In [4]:
### That's how to apply the generator to a state. It basically works as an array index. ###

def apply_gen(state, gen_name, generators):
    return state[generators[gen_name]]

new_state = apply_gen(central_state, "BL", generators)
print(f"new state: {new_state}", end="\n\n")

### Let's apply the inverse generator to the new_state to see if it goes back to the central_state ###
possibly_central_state = apply_gen(new_state, "-BL", generators)

is_back = np.all(central_state == possibly_central_state)
print(f"The state got back: {is_back}")


new state: [  0   1   2  16  17  15   6   7   8   9  10  11  12  13  14  25  26  24
   5   3   4  21  22  23  28  29  27  19  20  18  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  83  82
  72  73  71  70  76  77  78  79  80  81  99  98  74  75  86  87  88  89
  90  91  92  93  94  95  96  97  85  84 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119]

The state got back: True


Now let's access the test data from **test.csv**. It contains initial states of the puzzle, which were obtained by applying some random sequences of moves to the central_state

In [5]:
df_test = pd.read_csv(INPUTS_DIR/"test.csv", index_col = "initial_state_id")
df_test.head()

,initial_state
initial_state_id,
0,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18..."
1,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,22..."
2,"10,11,9,19,20,18,6,7,8,13,14,12,16,17,15,1,2,0..."
3,"0,1,2,3,4,5,32,30,31,9,10,11,12,13,14,15,16,17..."
4,"0,1,2,24,25,26,6,7,8,9,10,11,17,15,16,39,40,41..."


Finally, let's **load sample_submission.csv**. It contains some sample paths for the initial states from **test.csv** (mapped by initial_state_id).

In [6]:
df_sample_submission = pd.read_csv(INPUTS_DIR/"sample_submission.csv", index_col = "initial_state_id")
df_sample_submission.head()

,path
initial_state_id,
0,BR.U.BL.B.DR.B.D.-BL.FR.D.DL.FL.DL.BL.U.L.-F.-...
1,BR
2,BL.-L
3,R.-DR.BR
4,FR.-BL.BR.DL


## 	&#x1F914; Cheking the paths to be valid

In [7]:
### Some utilities ###

def parse_initial_state(inital_state_str: str) -> np.ndarray:
    return np.array([int(x) for x in inital_state_str.split(",")])
    
def parse_path(path_str: str) -> list[str]:
    return list(path_str.split("."))

def check_path(path_to_check: list[str], inital_state: np.ndarray, central_state: np.ndarray, generators: dict[str, np.ndarray]) -> bool:
    state = initial_state
    for gen_name in path_to_check:
        gen = generators[gen_name]
        state = state[gen]
    return np.all(state==central_state)
        

Now let's see if the path from the sample submission indeed leads to the central_state.

In [8]:
id_to_check = 1 # put any index between 0 and len(df_test) - 1
path_to_check = df_sample_submission.loc[id_to_check]["path"]
path_to_check = parse_path(path_to_check)
initial_state = df_test.loc[id_to_check]["initial_state"]
initial_state = parse_initial_state(initial_state)

path_is_valid = check_path(path_to_check, initial_state, central_state, generators)
print(f"Path for id {id_to_check} is valid: {path_is_valid}")

Path for id 1 is valid: True


## &#x1F526; Using beam_search from CayleyPy to imporve some paths

Well, the paths from sample_submission.csv won't get us far. Let's try to improve at least some of them using the beam_search algorithm from CayleyPy.

### Some info about Cayley graphs and CayleyPy:
[CayleyPy](https://github.com/cayleypy/cayleypy) is a python package designed to work with [Cayley graphs](https://en.wikipedia.org/wiki/Cayley_graph). Cayley graph is a mathematical structure useful to think about problems like ours. In our case, it is a graph (a set of nodes, some of which are connected with edges), where each node stands for a state of the puzzle. If a state can be achieved from another one, these two states are connected with an edge. In our case, this process is always reversible (as we include inverse generators), so the edges are undirected. With this setting, finding an optimal solution of the puzzle is equivalent to finding the shortest path on the corresponding Cayley graph.

Even for relatively small puzzles, Cayley graphs can be huge — e.g. for a 3x3x3 Rubik's cube, it is around 4.33E+19. For our master tetraminx, it is somewhat like 1.54E+32. There is little hope to keep the whole structure in RAM, let alone finding an optimal path with brute force.

[CayleyPy](https://github.com/cayleypy/cayleypy) package provides utilities to work with Cayley graphs. It is still under intensive development and already has some handy utilities like breadth-first search (BFS), beam search, random walks, and more than that. It also has a library for some puzzles (including this one). Feel free to explore the package, open an issue, or a pull request.



In [9]:
# installing cayleypy -- a library to work with cayley graphs 
!pip install git+https://github.com/cayleypy/cayleypy

  Cloning https://github.com/cayleypy/cayleypy to /tmp/pip-req-build-3awpltf5
  Running command git clone --filter=blob:none --quiet https://github.com/cayleypy/cayleypy /tmp/pip-req-build-3awpltf5
  Resolved https://github.com/cayleypy/cayleypy to commit 4577b9807e23bc4f2943ac07ba9596d5779c4d0a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for cayleypy: filename=cayleypy-0.1.0-py3-none-any.whl size=588320 sha256=a96a3ffd0df072f7a8d3d20766577fe975ec3017bc0d0bde21885169960a9fb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-2sd6n_na/wheels/5d/b5/53/86782f2010b218369465580e31a6c289b7f2a73390b39a23f3
Successfully built cayleypy


In [10]:
import torch
from cayleypy import CayleyGraphDef, CayleyGraph, Predictor

### Defining a Cayley graph in CayleyPy ###
# We will define the puzzle from scratch using the central_state and generators from puzzle_info.json

gens_names = list(generators.keys())
graph_def = CayleyGraphDef.create(
    generators = [generators[x] for x in gens_names],
    generator_names = gens_names,
    central_state = central_state
)

graph = CayleyGraph(graph_def)

/usr/local/lib/python3.11/dist-packages/cayleypy/cayley_graph.py:95: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.permutations_torch = torch.tensor(


Now let's find a path for a given initial_state with [Beam Search](https://en.wikipedia.org/wiki/Beam_search)  implemented in CayleyPy. The key component of beam search is a Predictor — an entity which estimates how "good" a given state is in terms of reaching some goal (in our case, finding the central_state). A Predictor can be built on top of some rules or a learned model, e.g. one estimating the distance from a given state to the central_state. A basic beam search algorithm based on a Predictor looks somewhat like this:

&#x1F913;**Beam search**&#x1F913;

Given an initial state $x_0$, the central state $x_c$, the goodness predictor $p(x)$, beam width parameter $W$, and maximum steps parameter $L$, do the following:

1. Define buffer, which keeps the pairs of paths and the goodnesses of the their last elements: $\mathcal{B} = \left\{\left([x_0,...,x_{k_i}^i], p(x_{k_i}^i)\right)\right\}$
2. Define and set the counter $i$ as zero.
3. Take $x_0$, calculate its goodness. Set the corresponding path as [$x_0$]. Put the pair $\left([x_0],p(x_0)\right)$ into $\mathcal{B}$. 
4. Take all paths from $\mathcal{B}$. Take the last states from each path. For each last state find their neighbors.
5. Apply predictor $p(x)$ to all recently found neighbors, get their goodnesses. Append the neighbors to the corresponding paths, put the new path-goodness pairs into $\mathcal{B}$.
6. If the number of paths in $\mathcal{B}$ is larger than $W$, drop all the states with the lowest goodness (break the ties randomly).
7. Check if any path in $\mathcal{B}$ ends up in $x_c$. If so, return this path.
8. If a valid path is not found, and if $i < L$, increase $i$ by 1 and go to step 4. Otherwise, return $None$ (acknowledge the failure).


Note, that there is a number of trivial and non-trivial optimizations which can be used to imporove this algorithm -- some of them are already in CayleyPy, while others are yorus to implement and use.

The success of beam search is mostly defined by a predictor $p(x)$; however, increasing the beam width $W$ and the number of steps $L$ can mitigate the predictor's imperfections to some extent. On the downside, increasing $W$ and $L$ are usually result in increaed memory consumoption and the running time.



CayleyPy has its own implementation of beam search. For demonstration purposes, we will use a simple rule called [Hamming distance](https://en.wikipedia.org/wiki/Hamming_distance), which basically counts the number of misplaced elements between the initial and central states. The lower the Hamming distance, the "better" the state. This rule in general is not really good for solving tasks like ours; however, it is simple to implement and fast to run. Let's check if beam search is able to find a path for some state with Hamming distance:

In [11]:
from tqdm.auto import tqdm
import torch
from cayleypy import CayleyGraphDef, CayleyGraph, Predictor

id_to_check = 8 # put any index between 0 and len(df_test) - 1
path_to_check = df_sample_submission.loc[id_to_check]["path"]
initial_state = df_test.loc[id_to_check]["initial_state"]
initial_state = parse_initial_state(initial_state)

beam_width = 1000
max_steps = 50
beam_search_result = graph.beam_search(
    start_state=initial_state,
    beam_width= beam_width,
    max_steps=max_steps,
    predictor=Predictor(graph, "hamming"),
    beam_mode="simple",
    return_path=True,
)

print(f"Path found: {beam_search_result.path_found}, beam_search length: {len(beam_search_result.path)}, sample path length: {len(path_to_check.split('.'))}")
bs_path = beam_search_result.get_path_as_string()
print(f"Beam search path: {bs_path}")
bs_path_is_valid = check_path(bs_path.split("."), initial_state, central_state, generators)
print(f"Beam_search path is valid: {bs_path_is_valid}")

Path found: True, beam_search length: 6, sample path length: 8
Beam search path: DR.F.-FR.-DR.-L.-L
Beam_search path is valid: True


Yay, it worked &#x1F601; Morevover, it is shorter than a path from **sample_submission.csv**.
Let's try to run beam search with hamming on the first few states.

In [12]:
## Feel free to play with the parameters and  process more states ###

beam_width = 1000
max_steps = 100
first_n_states_from_test = 15

final_paths = []
for row_id, row in tqdm(df_test[:first_n_states_from_test].iterrows(), total=len(df_test[:first_n_states_from_test])):
    sample_submission_path = df_sample_submission.loc[row_id]["path"]
    sample_submission_path = parse_path(sample_submission_path)
    initial_state = row["initial_state"]
    initial_state = parse_initial_state(initial_state)
    
    beam_search_result = graph.beam_search(
        start_state=initial_state,
        beam_width= beam_width,
        max_steps=max_steps,
        predictor=Predictor(graph, "hamming"),
        beam_mode="simple",
        return_path=True,
    )
    len_sample = len(sample_submission_path)
    if beam_search_result.path_found:
        len_bs = len(beam_search_result.path)
        bs_success_str = "True "
    else:
        len_bs = np.inf
        bs_success_str = "False"
        
    if  len_sample < len_bs:
        final_paths.append(".".join(sample_submission_path))
    else:
        final_paths.append(beam_search_result.get_path_as_string())
        
    print(f"{row_id:>3}. Beam search success: {bs_success_str}, beam_search len: {len_bs:<3}, sample_submission len: {len_sample:<3}")
        

  0%|          | 0/15 [00:00<?, ?it/s]

  0. Beam search success: False, beam_search len: inf, sample_submission len: 72 
  1. Beam search success: True , beam_search len: 1  , sample_submission len: 1  
  2. Beam search success: True , beam_search len: 2  , sample_submission len: 2  
  3. Beam search success: True , beam_search len: 3  , sample_submission len: 3  
  4. Beam search success: True , beam_search len: 4  , sample_submission len: 4  
  5. Beam search success: True , beam_search len: 5  , sample_submission len: 5  
  6. Beam search success: True , beam_search len: 6  , sample_submission len: 6  
  7. Beam search success: True , beam_search len: 7  , sample_submission len: 7  
  8. Beam search success: True , beam_search len: 6  , sample_submission len: 8  
  9. Beam search success: False, beam_search len: inf, sample_submission len: 9  
 10. Beam search success: False, beam_search len: inf, sample_submission len: 10 
 11. Beam search success: False, beam_search len: inf, sample_submission len: 11 
 12. Beam search


> **Hint**: the paths in the **sample_submission.csv** are the inverses of paths which were used to obtain the initial states. You can see, that the length of these paths goes from 1 to 1000 -- which means that the first few paths must close to the central_state. Solving them is relatively easy, but they are useful for santiy checks and debugging.



In [13]:
new_paths = final_paths + list(df_sample_submission[first_n_states_from_test:]["path"])
df_hamming_submission = df_sample_submission.copy()
df_hamming_submission["path"] = new_paths
df_hamming_submission.to_csv("submission.csv")